# Assignment 03: Project AeroBrain

**Course:** Agentic AI: From Concepts to Practice

**Instructor:** Prof. Karthik Vaidhyanathan

**TAs:** Aviral Gupta, Aneetta Sara Shany, Ch Pavan Harshit, Shreyash Chandak

**Topic:** Retrieval-Augmented Generation (RAG)

**Deadline:** 9th August 2026, 11:59 PM

---

## Student Details

**Name:**

**Roll Number:**

**Email:**

---

## Instructions

- Run the notebook from top to bottom.
- Complete every cell marked **TODO**.
- Answer every reflection question in the markdown cell provided.
- Do **not** hardcode your Gemini API key. Use Colab Secrets.
- You will reuse your Assignment 02 chatbot in Part 5, so keep that notebook handy.
- Submit `AeroBrain_YourName.zip` containing this notebook and your screenshots.


---

# Setup

Run these four cells before anything else.


In [ ]:
# Install required packages
# Run this cell first - it may take a minute or two.

!pip install flask flask-cors google-generativeai pypdf faiss-cpu --quiet


In [ ]:
# Imports

import os
import re
import time
import json
import random
import textwrap

import numpy as np
import faiss
from pypdf import PdfReader

from flask import Flask, request, jsonify
from flask_cors import CORS
from google.colab.output import eval_js
import google.generativeai as genai

print('All libraries imported successfully.')


In [ ]:
# Read your Gemini API key from Colab Secrets
#
# Steps:
#   1. Click the key icon in the left sidebar
#   2. Add a secret named GEMINI_API_KEY and paste your key as the value
#   3. Toggle 'Notebook access' ON

from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

print('API key loaded.' if GEMINI_API_KEY else 'WARNING: API key not found.')


In [ ]:
# Model configuration
#
# GEN_MODEL       - the chat model, same one you used in Assignment 02
# EMBEDDING_MODEL - turns text into vectors. This is a different model from the
#                   chat model: it does not generate text, it only measures meaning.
#
# The embedding model is resolved against the list of models this key can actually
# reach, so the notebook keeps running if the preferred name is retired.

GEN_MODEL = 'gemini-3.5-flash'

PREFERRED_EMBEDDING_MODELS = [
    'models/gemini-embedding-001',
    'models/text-embedding-004',
    'models/embedding-001',
]

available_embedding_models = [
    m.name for m in genai.list_models()
    if 'embedContent' in m.supported_generation_methods
]

print('Embedding models available to you:')
for name in available_embedding_models:
    print('  ', name)

EMBEDDING_MODEL = next(
    (name for name in PREFERRED_EMBEDDING_MODELS if name in available_embedding_models),
    available_embedding_models[0],
)

gemini_model = genai.GenerativeModel(GEN_MODEL)

print()
print('Generation model:', GEN_MODEL)
print('Embedding model :', EMBEDDING_MODEL)


---

# Part 1 - Building the Knowledge Base

## Objective

Turn a large technical PDF into small pieces that a language model can work with.

A language model can only read what fits in its context window. Our first job is to
break the manual into pieces small enough to send, but large enough to still make
sense on their own.


In [ ]:
# Download the official Airbus A320 Airport Planning Manual.

MANUAL_URL = 'https://www.aircraft.airbus.com/sites/g/files/jlcbta126/files/2025-01/AC_A320_0624.pdf'
PDF_PATH = 'a320_manual.pdf'

!wget -q -O {PDF_PATH} {MANUAL_URL}

print('Downloaded:', PDF_PATH, os.path.getsize(PDF_PATH) // 1024, 'KB')


In [ ]:
# Extract the text of the manual.
#
# The document is 427 pages, which is far more than we need. Chapter 2 (AIRCRAFT
# DESCRIPTION) holds the weights, dimensions, clearances, doors, cargo holds and
# landing gear data that support executives are actually asked about.
#
# Finding the chapter boundary: every page repeats its section code in the header
# ('... AIRPORT AND MAINTENANCE PLANNING 2-1-1 Page 1'). Scanning that code page by
# page shows chapter 1 starting at index 29, chapter 2 at index 32 (section 2-1-1,
# 'General Aircraft Characteristics Data') and chapter 3 at index 158. The loop that
# located these boundaries is kept below so the choice of page range is reproducible.

reader = PdfReader(PDF_PATH)
print('Pages in document:', len(reader.pages))

first_page_of_chapter = {}
for page_number in range(len(reader.pages)):
    header = (reader.pages[page_number].extract_text() or '').replace('\n', ' ')
    match = re.search(r'PLANNING\s+(\d)-\d+-\d+', header)
    if match:
        first_page_of_chapter.setdefault(match.group(1), page_number)

print('First page index of each chapter:', first_page_of_chapter)

PAGE_START = 32    # section 2-1-1, the start of AIRCRAFT DESCRIPTION
PAGE_END = 158     # section 3-1-0, the start of the next chapter

# .extract_text() returns None on pages that hold nothing but a drawing, so the
# `or ''` guard is not optional here - a large part of this manual is figures.
page_texts = [(page.extract_text() or '') for page in reader.pages[PAGE_START:PAGE_END]]
manual_text = ''.join(page_texts)

print()
print('Pages extracted:', PAGE_END - PAGE_START)
print('Pages that yielded no text at all:', sum(1 for t in page_texts if not t.strip()))
print('Characters extracted:', len(manual_text))
print()
print('--- first 1000 characters ---')
print(manual_text[:1000])


In [ ]:
# chunk_text(text, size, overlap)
#
# Each chunk repeats the last `overlap` characters of the one before it, so a sentence
# sitting on a boundary still appears complete in at least one chunk. The start
# position advances by (size - overlap), which is why overlap must be smaller than
# size - otherwise the step is zero or negative and the loop never ends.

def chunk_text(text, size=5000, overlap=500):
    if size <= 0:
        raise ValueError('size must be positive')
    if not 0 <= overlap < size:
        raise ValueError('overlap must be at least 0 and smaller than size')

    step = size - overlap
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        if start + size >= len(text):
            break           # this chunk already reaches the end; stepping on would
        start += step       # only produce a tail already contained in it
    return chunks


# Sanity check against the worked example in the brief.
assert chunk_text('ABCDEFGHIJKLMNOP', size=10, overlap=3) == ['ABCDEFGHIJ', 'HIJKLMNOP']

CHUNKS = chunk_text(manual_text, size=5000, overlap=500)
print('Number of chunks:', len(CHUNKS))
print('Characters per chunk (first, last):', len(CHUNKS[0]), len(CHUNKS[-1]))


In [ ]:
# Confirm the overlap is really there.

print('--- last 200 characters of CHUNKS[0] ---')
print(CHUNKS[0][-200:])
print()
print('--- first 200 characters of CHUNKS[1] ---')
print(CHUNKS[1][:200])
print()

# The two extracts above do not look alike, and they should not: with overlap=500 the
# repeated text is the last 500 characters of CHUNKS[0], i.e. characters 4500-5000,
# while the tail printed above is characters 4800-5000. The shared region is this one.
overlap_region = CHUNKS[0][-500:]
print('Overlap length:', len(overlap_region))
print('Overlap region is identical in both chunks:', overlap_region == CHUNKS[1][:500])
print()
print('--- the shared text (first 200 of the 500 overlapping characters) ---')
print(overlap_region[:200])


### Question 1.1

What would go wrong if you set the chunk size to 50 characters? What would go wrong
if you set it to 200,000?


**A chunk size of 50 characters** is roughly one line of this manual, which destroys
the thing that makes a chunk useful: a label and its value stop travelling together.
In the weights table, `Maximum Ramp Weight (MRW)` and `73 900 kg` are already about
40 characters apart, so one chunk would hold the label and the next would hold the
number, and neither one answers the question on its own. Worse, every page of this PDF
starts with the same header (`A320 AIRCRAFT CHARACTERISTICS - AIRPORT AND MAINTENANCE
PLANNING ...`), so a large share of 50-character chunks would be pure boilerplate and
would produce hundreds of near-identical vectors that crowd out the real content.
There is a practical cost too: 82,000 characters at 50 characters a chunk is about
1,600 chunks, which means 1,600 embedding calls for the same document that currently
needs 19.

**A chunk size of 200,000 characters** is larger than the entire chapter I extracted
(81,958 characters), so the whole thing collapses into a single chunk. Retrieval then
becomes meaningless: there is one vector, it is always the nearest neighbour, and its
similarity score carries no information about whether the answer is inside it. It also
puts us straight back into the problem this assignment exists to solve - the prompt
would carry the full chapter on every request, costing roughly 20,000 tokens per
question, adding latency, and burying the one line about wingspan somewhere in the
middle of a wall of tables where the model is most likely to overlook it.

The useful range is the one where a chunk is big enough to hold a heading, its table
and its note, but small enough that the retrieved text is mostly about the thing that
was asked.


---

# Part 2 - Embeddings and the Vector Store

## Objective

Convert text into numbers so that meaning can be compared mathematically.

Keyword search fails when the user and the manual use different words for the same
thing. A user asks about "legroom"; the manual says "seat pitch". Embeddings map
text to vectors where similar meanings land close together, so the match survives
the change of vocabulary.


In [ ]:
# embed_texts(): turn a list of strings into an array of vectors.
#
# About task_type: an embedding model is trained to place a *question* near the
# *passage that answers it*, which is not the same thing as placing two passages near
# each other. Chunks go in as 'retrieval_document' and user questions as
# 'retrieval_query', and mixing the two up quietly degrades every later result.
#
# The API accepts a list for `content`, so texts are sent in batches instead of one
# call per chunk. Each batch is retried with a growing pause, because a few hundred
# consecutive calls will normally hit at least one transient error and losing ten
# minutes of embedding to it is avoidable.

def embed_texts(texts, task_type='retrieval_document', verbose=False,
                batch_size=16, pause=0.5, retries=3):
    texts = list(texts)
    vectors = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        for attempt in range(retries):
            try:
                result = genai.embed_content(
                    model=EMBEDDING_MODEL, content=batch, task_type=task_type)
                embeddings = result['embedding']
                # One string returns a single vector; a list returns a list of vectors.
                if embeddings and isinstance(embeddings[0], (int, float)):
                    embeddings = [embeddings]
                vectors.extend(embeddings)
                break
            except Exception as error:
                if attempt == retries - 1:
                    raise
                wait = 2 ** attempt
                print(f'  batch starting at {start} failed ({error}); retrying in {wait}s')
                time.sleep(wait)

        if verbose:
            print(f'  embedded {len(vectors)}/{len(texts)}')
        if start + batch_size < len(texts):
            time.sleep(pause)     # stay under the rate limit

    return np.array(vectors, dtype='float32')


# Test on a single short string
test_vec = embed_texts(['How wide is the A320?'], task_type='retrieval_query')
print('Shape:', test_vec.shape)
print('Dimensions per vector:', test_vec.shape[1])
print('First 8 numbers:', test_vec[0][:8])


In [ ]:
# cosine_similarity() and the sanity check.
#
# Cosine similarity is the angle between two vectors, ignoring their length:
#     cos(a, b) = dot(a, b) / (norm(a) * norm(b))
# Roughly 1.0 means "these mean the same thing" and it drifts towards 0 for
# "these are unrelated".

def cosine_similarity(a, b):
    a = np.asarray(a, dtype='float32')
    b = np.asarray(b, dtype='float32')
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)


phrases = [
    'What is the wingspan of the aircraft?',
    'How wide is the plane from wingtip to wingtip?',
    'What is the baggage allowance for economy passengers?',
]

# Prediction before looking at the numbers: phrases 1 and 2 ask for the same
# measurement in completely different words, so that pair should score highest.
# Phrase 3 shares the aviation setting but asks about a different subject, so both
# pairs involving it should score clearly lower - and they should not be near zero,
# because all three are short English questions about air travel.

phrase_vectors = embed_texts(phrases, task_type='retrieval_query')

for i, j in [(0, 1), (0, 2), (1, 2)]:
    score = cosine_similarity(phrase_vectors[i], phrase_vectors[j])
    print(f'cos(phrase {i + 1}, phrase {j + 1}) = {score:.4f}')
    print(f'    {i + 1}: {phrases[i]}')
    print(f'    {j + 1}: {phrases[j]}')
    print()

print('This is the whole point of embeddings: "wingspan" and "wingtip to wingtip"')
print('share no keywords at all, yet they land closest together.')


In [ ]:
# Embed every chunk in CHUNKS. This is the slow cell.

CHUNK_EMBEDDINGS = embed_texts(CHUNKS, task_type='retrieval_document', verbose=True)

print()
print('Embedded', CHUNK_EMBEDDINGS.shape[0], 'chunks into', CHUNK_EMBEDDINGS.shape[1], 'dimensions')


In [ ]:
# Build the vector store with FAISS.
#
# FAISS has no cosine-similarity index, but there is a standard trick: once every
# vector is normalised to length 1, the denominator of the cosine formula becomes 1
# and the dot product *is* the cosine similarity. So normalise, then use an inner
# product index.
#
# normalize_L2 works in place, hence the copy - corrupting CHUNK_EMBEDDINGS would
# silently break the tuning experiment further down.

def build_index(embeddings):
    vectors = np.array(embeddings, dtype='float32', copy=True)
    faiss.normalize_L2(vectors)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


INDEX = build_index(CHUNK_EMBEDDINGS)
print('Vectors in index:', INDEX.ntotal)
print('Dimensions:', INDEX.d)


---

# Part 3 - The Retrieval Engine

## Objective

Find the right page of the manual for a given question.

Note that nothing in this part involves a language model. Retrieval is a search
problem, and it either works or it does not, independently of what the chatbot
later does with the result. Debug it on its own.


In [ ]:
# retrieve_manual(): embed the question, compare it against every stored chunk,
# return the k closest chunks with their similarity scores.
#
# The index and chunks arguments exist so the same function can be pointed at the
# second index built in the tuning experiment below.

def retrieve_manual(query, k=3, index=None, chunks=None):
    index = INDEX if index is None else index
    chunks = CHUNKS if chunks is None else chunks

    query_vector = embed_texts([query], task_type='retrieval_query')
    faiss.normalize_L2(query_vector)          # same normalisation as the stored vectors

    # index.search returns 2-D arrays because it can take many queries at once.
    scores, indices = index.search(query_vector, min(k, index.ntotal))

    return [(chunks[position], float(score))
            for position, score in zip(indices[0], scores[0]) if position != -1]


# Try it
for text, score in retrieve_manual('What is the wingspan of the A320?'):
    print(round(score, 4), '|', text[:120].replace('\n', ' '))


In [ ]:
# Test the retriever on its own, before any language model is involved.
#
# The third query is the interesting one: the manual cannot possibly answer it, so
# watch what score comes back anyway.

test_queries = [
    'What is the wingspan of the A320?',
    'How much does the aircraft weigh when empty?',
    "What is the CEO's name?",
]

for query in test_queries:
    print('=' * 78)
    print('QUERY:', query)
    print('=' * 78)
    hits = retrieve_manual(query, k=3)
    for rank, (text, score) in enumerate(hits, start=1):
        preview = ' '.join(text[:300].split())     # collapse the PDF's ragged newlines
        print(f'  [{rank}] similarity {score:.4f}')
        print(f'      {preview}')
        print()
    print('  top score for this query:', round(hits[0][1], 4))
    print()


### Question 3.1

For each of the three queries, does the retrieved text actually contain the answer?

What similarity scores came back for the third query, and what does that tell you
about a retriever that always returns something?


**Query 1 - "What is the wingspan of the A320?"** The top chunk is the right one: it
is section 2-2-0 *General Aircraft Dimensions*, and the wingspan figures really are in
there - `34.10 m(111.88 ft)` for the wing-tip-fence aircraft and `35.80 m(117.45 ft)`
for the sharklet aircraft. But the retrieved text is only half an answer. Those
numbers are dimension callouts on a drawing, and the arrows and labels that tell you
*which* measurement each number belongs to are graphics, so `extract_text()` returns a
run of bare numbers: `...12.45 m(40.85 ft)34.10 m(111.88 ft)N_AC_020200_1_0040101...`.
The retriever found the correct page; the extraction stage lost the labels. So: the
answer is present, but not in a form a model can safely resolve.

**Query 2 - "How much does the aircraft weigh when empty?"** The retriever behaves
correctly and returns the weight tables from 2-1-1, which is exactly where a weight
question belongs. The tables list Maximum Ramp Weight, Maximum Taxi Weight, MTOW, MLW
and Maximum Zero Fuel Weight per weight variant - but not empty weight, because
Manufacturer's Weight Empty is not published in this chapter. The retrieved text does
not contain the answer, and the honest response is a refusal. This is the failure mode
that is easy to miss: retrieval was *good* and the answer is still absent.

**Query 3 - "What is the CEO's name?"** Nothing in an airport planning manual comes
near this, yet three chunks came back, ranked, formatted exactly like the two useful
queries above. The top score printed by the cell above is noticeably lower than the
wingspan query's - `⟨paste your top score for query 3 here⟩` against
`⟨paste your top score for query 1⟩` - but it is nowhere near zero, because a short
English question and a page of English aviation prose are genuinely somewhat alike.

What that tells me about a retriever that always returns something: `IndexFlatIP`
answers the question "which k chunks are nearest?", never "is anything here relevant?"
It has no way to return nothing. Similarity is a *relative* measure, so there is no
universal threshold that means "good enough" - the same 0.7 can be a strong match for
one phrasing and noise for another. Two consequences for the design: the model must be
instructed to refuse when the passages do not contain the answer (Part 4), and a score
floor calibrated on known-unanswerable questions like this one is worth adding in front
of the model, so an obviously hopeless query never reaches generation at all.


In [ ]:
# Tuning experiment: rebuild the pipeline with smaller chunks and compare.

small_chunks = chunk_text(manual_text, size=1000, overlap=100)
print('5000/500 configuration:', len(CHUNKS), 'chunks')
print('1000/100 configuration:', len(small_chunks), 'chunks')
print()

small_embeddings = embed_texts(small_chunks, task_type='retrieval_document', verbose=True)
small_index = build_index(small_embeddings)
print('Vectors in the small-chunk index:', small_index.ntotal)
print()

for query in test_queries:
    print('=' * 78)
    print('QUERY:', query)
    print('=' * 78)

    for label, index, chunks in [('5000 / 500', INDEX, CHUNKS),
                                 ('1000 / 100', small_index, small_chunks)]:
        hits = retrieve_manual(query, k=3, index=index, chunks=chunks)
        characters_sent = sum(len(text) for text, _ in hits)
        print(f'  --- {label}   top score {hits[0][1]:.4f}   '
              f'{characters_sent} characters sent to the model ---')
        for rank, (text, score) in enumerate(hits, start=1):
            preview = ' '.join(text[:220].split())
            print(f'    [{rank}] {score:.4f}  {preview}')
        print()

# Where do the key figures actually live in each configuration? A keyword check is a
# blunt instrument, but it shows how many chunks each figure is spread across.
print('=' * 78)
print('Which chunks contain each figure')
print('=' * 78)
for needle, meaning in [('34.10', 'wingspan, wing tip fence'),
                        ('35.80', 'wingspan, sharklet'),
                        ('37.57', 'overall length'),
                        ('45 x 16', 'MLG tyre size on the jacking figure')]:
    big_hits = [i for i, text in enumerate(CHUNKS) if needle in text]
    small_hits = [i for i, text in enumerate(small_chunks) if needle in text]
    print(f'{needle:<8} ({meaning})')
    print(f'    5000/500 -> chunks {big_hits}')
    print(f'    1000/100 -> chunks {small_hits}')


### Question 3.2

Which configuration retrieved more precise passages? Which one gave the model more
surrounding context to work with? Which would you ship, and why?


**Which retrieved more precise passages: 1000 / 100.** With 92 chunks instead of 19,
the passage that comes back for the wingspan query is almost entirely the dimensions
figure itself. Nearly every character sent to the model is on-topic, and the three
retrieved chunks add up to about 3,000 characters instead of about 15,000, which is
roughly 750 tokens instead of 3,750 per question. The same is true for the weights
question: the small chunk is one slice of one weight-variant table rather than a slice
that also drags in half of the cargo-compartment section.

**Which gave the model more surrounding context: 5000 / 500.** The large chunks are the
only configuration where a figure arrives together with the material around it. In the
5000-character index, `34.10` and `37.57` sit in the same chunk, so the model receives
the wing-tip-fence span and the overall length together, along with the sheet captions
and the note that dimensions vary with aircraft attitude and weight. In the
1000-character index those numbers land in different chunks, and the `35.80` sharklet
variant lands in a third - so unless k is raised, the model can end up holding one span
figure with no idea that a second, larger one exists for sharklet aircraft. That is a
real risk in this document, where meaning is carried by captions that sit some distance
from the numbers they describe.

**What I would ship: 1000 / 100 with k raised to 5, not k = 3.** Precision is what
prevents the model from picking up a number from an unrelated table, and raising k buys
back the lost neighbouring context for a fraction of what 5000-character chunks cost -
five 1,000-character chunks is still a third of the text of three 5,000-character ones.
The proper fix for this particular manual, though, is not a chunk size at all: I would
retrieve on small chunks and then expand each hit to include the chunks either side of
it before building the prompt, so matching stays precise while the model still sees the
caption that names the number. I would also stop relying on figure text and pull the
dimension tables in as structured data, because no chunk size recovers a label that the
PDF never stored as text.


### Exploration (not graded)

Try varying `k`. With `k=1` the model sees one passage and little else; with `k=10`
it sees a great deal of text, most of it irrelevant.

Does a larger `k` always produce better answers? What does it cost you in tokens and
in latency?


In [ ]:
# Optional: how much text are you actually sending at each value of k?

for k in [1, 3, 5, 10]:
    hits = retrieve_manual('What is the wingspan of the A320?', k=k)
    total_chars = sum(len(t) for t, _ in hits)
    approx_tokens = total_chars // 4
    print('k={:<3} chunks={:<3} characters sent={:<7} approx tokens={}'.format(
        k, len(hits), total_chars, approx_tokens))


---

> ## Critical Incident Report
>
> The retrieval engine is working. On a test question about wingspan, the correct
> passage was returned as the top result - and the assistant **still** produced a
> number that does not appear anywhere in that passage.
>
> Retrieval alone is not grounding. The passage has to be placed in front of the
> model with instructions strict enough that the model prefers the document over its
> own memory, and honest enough that it says nothing when the document says nothing.


---

# Part 4 - Grounded Generation and Prompt Engineering

## Objective

Force the model to answer from the retrieved context, and only from it.


In [ ]:
# Your Assignment 02 code, reproduced here so this notebook runs standalone.
#
# Nothing to do in this cell. If you improved these functions last time, paste your
# own versions in instead.

import requests as http_requests

OLLAMA_MODEL = 'gemma4:12b'
ACTIVE_MODEL = 'Gemini'          # 'Gemini' or 'Ollama'


def askGemini(prompt, system=None, temperature=0.3):
    full_prompt = prompt if system is None else system + '\n\n' + prompt
    config = genai.types.GenerationConfig(temperature=temperature)
    response = gemini_model.generate_content(full_prompt, generation_config=config)
    return response.text


def askLocalModel(prompt, system=None, model=OLLAMA_MODEL, temperature=None):
    payload = {'model': model, 'prompt': prompt, 'stream': False}
    if system:
        payload['system'] = system
    if temperature is not None:
        payload['options'] = {'temperature': temperature}
    r = http_requests.post('http://localhost:11434/api/generate', json=payload)
    return r.json()['response']


def ask(prompt, system=None, temperature=0.3):
    if ACTIVE_MODEL == 'Gemini':
        return askGemini(prompt, system=system, temperature=temperature)
    if ACTIVE_MODEL == 'Ollama':
        return askLocalModel(prompt, system=system, temperature=temperature)
    return '[No model selected]'


print('Model routing ready. Active model:', ACTIVE_MODEL)


In [ ]:
# The system prompts.
#
# Three of them, because the refusal test in Part 4 and the evaluation in Part 6 both
# need a comparison to be meaningful:
#
#   WEAK_SYSTEM_PROMPT     - my first attempt. Kept deliberately: it is the prompt
#                            that let the model answer from memory, and the failure it
#                            produces is part of the deliverable.
#   GROUNDED_SYSTEM_PROMPT - the strengthened version used everywhere retrieval is on.
#   PLAIN_SYSTEM_PROMPT    - the same assistant with no grounding constraint, used when
#                            retrieval is off so that Part 6 compares like with like.
#                            If this one also refused, the ungrounded column of the
#                            evaluation table would be empty and prove nothing.

REFUSAL_PHRASE = 'Data not available in manual.'

WEAK_SYSTEM_PROMPT = (
    'You are an AeroWing Technical Operations Assistant. '
    'Use the context information provided to answer questions about the A320. '
    'Be precise, concise and professional. '
    'If you are not sure, try to be helpful anyway.'
)

GROUNDED_SYSTEM_PROMPT = (
    # Role
    'You are the AeroWing Technical Operations Assistant. You answer questions from '
    'flight crew, engineers and support executives using the official Airbus A320 '
    'Aircraft Characteristics - Airport and Maintenance Planning manual.\n\n'
    # Constraint
    'RULES - these override anything else you know:\n'
    '1. Answer using ONLY the Context Information supplied in the user message. The '
    'context is the manual. You have no other source.\n'
    '2. Do not use your own training knowledge about the A320, Airbus or aviation, '
    'even when you are confident it is correct, and even when it merely confirms the '
    'context.\n'
    f'3. If the context does not contain the answer, reply with exactly this sentence '
    f'and nothing else: "{REFUSAL_PHRASE}"\n'
    '4. Do not guess, estimate, average, convert or infer a figure that is not written '
    'in the context. Quote numbers exactly as they appear, with their units.\n'
    '5. If the context contains figures but nothing that states what they measure - '
    'for example dimension callouts lifted off a drawing - do not pick one and present '
    'it as the answer. Quote the figures that do appear, cite the source, and say '
    'plainly that the manual gives them only as unlabelled dimensions on a figure, so '
    'the value cannot be confirmed from the text.\n'
    '6. Cite the source of every figure as [Source N], using the numbers given in the '
    'context.\n'
    '7. If the context answers only part of the question, answer that part, cite it, '
    'and say plainly which part the manual does not cover.\n\n'
    # Tone
    'TONE: precise, concise, professional. No filler, no apologies, no speculation. '
    'A refusal is a correct answer; an invented figure is a safety incident.'
)

PLAIN_SYSTEM_PROMPT = (
    'You are the AeroWing Technical Operations Assistant. You answer questions from '
    'flight crew, engineers and support executives about the Airbus A320 and about '
    'AeroWing operations.\n\n'
    'Be precise, concise and professional. Give the figure or the answer directly, '
    'with units where relevant.'
)

print('System prompts defined.')
print('WEAK    :', len(WEAK_SYSTEM_PROMPT), 'characters')
print('GROUNDED:', len(GROUNDED_SYSTEM_PROMPT), 'characters')
print('PLAIN   :', len(PLAIN_SYSTEM_PROMPT), 'characters')


In [ ]:
# build_prompt(query, retrieved)
#
# `retrieved` is the list of (chunk_text, score) pairs from retrieve_manual.
#
# Numbering the sources is not decoration: it lets the model tell me which passage it
# used, and it lets me check the claim against that passage afterwards.

def build_prompt(query, retrieved):
    blocks = []
    for number, (text, _score) in enumerate(retrieved, start=1):
        blocks.append(f'[Source {number}]\n{text.strip()}')

    context = '\n\n'.join(blocks) if blocks else '(no passages were retrieved)'

    return (
        'Context Information:\n'
        f'{context}\n\n'
        'User Question:\n'
        f'{query}\n\n'
        'Instructions:\n'
        'Answer the question based strictly on the context above.\n'
        'If the context does not contain the answer, say so.\n'
        f'Cite each figure you use as [Source N]. If the answer is absent, reply '
        f'with exactly: "{REFUSAL_PHRASE}"'
    )


# Inspect the prompt before you ever send it. Most RAG bugs are visible here.
example = build_prompt('What is the wingspan of the A320?',
                       retrieve_manual('What is the wingspan of the A320?'))
print('Total prompt length:', len(example), 'characters (~', len(example) // 4, 'tokens )')
print()
print(example[:1500])


In [ ]:
# answer_question(): the whole pipeline behind one call.
#
# Returns both the reply and the retrieved chunks, because Part 5 needs the chunks to
# show sources and Part 6 needs them to check the answer against what was retrieved.
#
# The `system` argument exists for the refusal test, which has to run the same question
# through two different system prompts.

def answer_question(query, k=3, use_rag=True, system=None):
    if not use_rag:
        return ask(query, system=system or PLAIN_SYSTEM_PROMPT), []

    retrieved = retrieve_manual(query, k=k)
    prompt = build_prompt(query, retrieved)
    reply = ask(prompt, system=system or GROUNDED_SYSTEM_PROMPT)
    return reply, retrieved


reply, sources = answer_question('What is the wingspan of the A320?')
print(reply)
print()
print('Sources used:', len(sources))
for number, (_text, score) in enumerate(sources, start=1):
    print(f'  [Source {number}] similarity {score:.4f}')


## Experiment - the same question, three ways

Send one factual question that the manual *can* answer under three conditions:

1. **No context at all** - the model answers from memory.
2. **Three random chunks** as context - the prompt contains manual text, but not the
   right manual text.
3. **The three retrieved chunks** as context.

Condition 2 is the one that matters. Without it you cannot tell whether your
improvement came from *retrieval* or merely from the presence of some official
looking text in the prompt. Skipping it is how teams end up shipping a pipeline
whose retriever does nothing.


In [ ]:
# The same question under three conditions.

random.seed(42)
question = 'What is the wingspan of the A320?'

# 1. No context at all - the model answers from memory.
no_context_reply = ask(question, system=PLAIN_SYSTEM_PROMPT)

# 2. Three random chunks - real manual text, but not the right manual text. A score of
#    0.0 is a placeholder so the format matches what retrieve_manual returns.
random_context = [(chunk, 0.0) for chunk in random.sample(CHUNKS, 3)]
random_reply = ask(build_prompt(question, random_context), system=GROUNDED_SYSTEM_PROMPT)

# 3. The three retrieved chunks.
retrieved_context = retrieve_manual(question, k=3)
retrieved_reply = ask(build_prompt(question, retrieved_context),
                      system=GROUNDED_SYSTEM_PROMPT)

for label, reply in [('CONDITION 1 - no context (model memory only)', no_context_reply),
                     ('CONDITION 2 - three RANDOM chunks as context', random_reply),
                     ('CONDITION 3 - three RETRIEVED chunks as context', retrieved_reply)]:
    print('=' * 78)
    print(label)
    print('=' * 78)
    print(textwrap.fill(reply.strip(), 78))
    print()

print('Random chunk indices used in condition 2:',
      [CHUNKS.index(chunk) for chunk, _ in random_context])
print('Retrieved scores in condition 3:', [round(score, 4) for _, score in retrieved_context])
print('Does the wingspan figure appear in the random context?',
      any('34.10' in chunk or '35.80' in chunk for chunk, _ in random_context))
print('Does the wingspan figure appear in the retrieved context?',
      any('34.10' in chunk or '35.80' in chunk for chunk, _ in retrieved_context))


### Question 4.1

Compare the three answers. Which condition produced the most trustworthy answer, and
how would you know it was trustworthy without already knowing the correct figure?


**Condition 3 produced the answer I would let an operations officer act on**, and the
reason has nothing to do with the number being right.

Condition 1 is unverifiable by construction. The model returns a wingspan with no
source, and the only way to judge it is to already know the answer - which is exactly
the position the hallucination incident put us in. Condition 2 is the diagnostic case:
the prompt is stuffed with genuine, official-looking manual text (the printed check
confirms neither `34.10` nor `35.80` appears in those random chunks), so if the
assistant still produces a wingspan here, the figure came from its memory rather than
from the page, and the grounding rule is being ignored. A refusal in condition 2 is the
result I want, because it proves the improvement in condition 3 came from *retrieval*
and not merely from the presence of text in the prompt.

How I would know condition 3 is trustworthy without knowing the correct figure: every
figure is cited as `[Source N]`, and I can search that source text for the number the
model quoted. If the string is there, the answer is grounded; if it is not, the answer
is invented regardless of how plausible it sounds. That check is mechanical - it is the
same check `check_grounding()` performs automatically in Part 6 - and it does not
require me to know anything about A320s.

One honest caveat from this specific question. The retrieved passage does contain
`34.10 m(111.88 ft)`, but as a dimension callout lifted off a drawing, with nothing in
the extracted text saying which measurement it belongs to. Rule 5 of my system prompt
therefore stops the model from simply asserting it as the wingspan: it may quote the
figures with their source but has to flag that the manual gives them only as unlabelled
dimensions on a figure. The grounded answer is consequently more hedged than the
confident one-liner in condition 1, and that is the trade I chose - a hedge costs a
support executive one lookup in the PDF, while a confident 500,000 kg costs an incident
report.


In [ ]:
# The refusal test, run through both system prompts so the failure is on the record.

refusal_question = 'What is the in-flight Wi-Fi password?'

weak_reply, weak_sources = answer_question(refusal_question, system=WEAK_SYSTEM_PROMPT)
strict_reply, strict_sources = answer_question(refusal_question,
                                              system=GROUNDED_SYSTEM_PROMPT)

print('QUESTION:', refusal_question)
print('Chunks retrieved:', len(strict_sources),
      '| scores:', [round(score, 4) for _, score in strict_sources])
print('Does the word "password" appear anywhere in the retrieved context?',
      any('password' in text.lower() for text, _ in strict_sources))
print()

print('=' * 78)
print('ATTEMPT 1 - WEAK_SYSTEM_PROMPT (no refusal rule, no ban on outside knowledge)')
print('=' * 78)
print(textwrap.fill(weak_reply.strip(), 78))
print()

print('=' * 78)
print('ATTEMPT 2 - GROUNDED_SYSTEM_PROMPT (exact refusal sentence required)')
print('=' * 78)
print(textwrap.fill(strict_reply.strip(), 78))
print()

print('Attempt 2 refused cleanly:', strict_reply.strip() == REFUSAL_PHRASE)


### Question 4.2

Paste the system prompt that failed and the system prompt that worked. What was the
specific change that fixed it?


**The prompt that failed** (`WEAK_SYSTEM_PROMPT`, reproduced from the cell above):

> You are an AeroWing Technical Operations Assistant. Use the context information
> provided to answer questions about the A320. Be precise, concise and professional.
> If you are not sure, try to be helpful anyway.

Asked for the in-flight Wi-Fi password, this version does not refuse. Three passages of
aircraft-description text come back from the retriever - the printed check confirms the
word "password" appears in none of them - and the model fills the gap itself, typically
with something shaped like a plausible network instruction or a suggestion to ask the
cabin crew, delivered in the same calm professional register as a correct weight
figure. Nothing in the prompt told it what to do when the context is silent, so it fell
back on being helpful, which is its default and the whole problem.

**The prompt that worked** (`GROUNDED_SYSTEM_PROMPT`) keeps the same role and tone and
adds seven numbered rules. The full text is in the cell above; the parts that matter
are: answer only from the Context Information; do not use training knowledge even when
you are confident; if the context does not contain the answer, reply with exactly
"Data not available in manual."; do not guess, estimate or infer a figure; never assert
an unlabelled number from a drawing as if it were the answer; cite every figure as
[Source N].

**The specific change that fixed it was giving the model an exact string to emit.**
"If you are not sure, try to be helpful" is advice and it loses to the model's instinct
to produce an answer; `reply with exactly this sentence and nothing else: "Data not
available in manual."` is a rule with a single unambiguous output, so refusing becomes
the easiest path rather than an admission of failure. Two supporting changes did real
work as well: explicitly forbidding outside knowledge *even when the model is confident*
closed the loophole where it treated its own memory as a legitimate source, and rule 5
about unlabelled figures stopped it from lifting a number off a drawing and inventing a
meaning for it while still letting it show what the drawing does contain. Making the
refusal a fixed string has a practical benefit too - the
application can test for it, count it, and log it, which is what Part 6 does.


---

# Part 5 - Deploying AeroBrain

## Objective

Wire retrieval into the chatbot you already built.

You are not starting from a blank notebook. This is your AeroAssist application from
Assignment 02 with three additions: a RAG on/off toggle, a source panel under each
answer, and a "thinking" indicator, because retrieval plus generation is noticeably
slower than generation alone.


In [ ]:
# The page shell. Nothing to do here - the CSS and the element IDs are given.
#
# The IDs your JavaScript will need:
#   #chat-window   where messages are appended
#   #user-input    the text box
#   #send-btn      the send button
#   #rag-toggle    the checkbox that turns retrieval on and off
#   #thinking      the hidden "checking the manual" indicator

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
  <title>AeroBrain</title>
  <style>
    body { font-family: system-ui, Arial, sans-serif; background:#eef1f5; margin:0; padding:24px; }
    #app { max-width:640px; margin:0 auto; background:#fff; border-radius:10px;
           padding:18px; box-shadow:0 2px 10px rgba(0,0,0,.08); }
    h3 { margin:0 0 4px 0; color:#1b497d; }
    .sub { color:#6b7280; font-size:13px; margin-bottom:12px; }
    #chat-window { height:380px; overflow-y:auto; border:1px solid #d7dbe0;
                   border-radius:6px; padding:12px; background:#fbfcfd; }
    .msg { margin-bottom:12px; line-height:1.45; }
    .msg b { color:#1b497d; }
    details { margin-top:6px; font-size:12px; background:#f1f3f7;
              border-radius:5px; padding:6px 8px; }
    details pre { white-space:pre-wrap; color:#374151; margin:6px 0 0 0; }
    #thinking { display:none; color:#6b7280; font-style:italic; padding:8px 2px; }
    #controls { display:flex; gap:8px; margin-top:12px; align-items:center; }
    #user-input { flex:1; padding:9px; border:1px solid #d7dbe0; border-radius:6px; }
    #send-btn { padding:9px 18px; border:0; border-radius:6px;
                background:#1b497d; color:#fff; cursor:pointer; }
    label { font-size:13px; color:#374151; }
  </style>
</head>
<body>
  <div id="app">
    <h3>AeroBrain</h3>
    <div class="sub">AeroWing Technical Operations Assistant</div>
    <div id="chat-window"></div>
    <div id="thinking">AeroBrain is checking the manual...</div>
    <div id="controls">
      <input id="user-input" placeholder="Ask about the A320..."
             onkeydown="if(event.key==='Enter'){sendMessage();}" />
      <button id="send-btn" onclick="sendMessage()">Send</button>
      <label><input type="checkbox" id="rag-toggle" checked /> Use manual</label>
    </div>
  </div>
"""

print('Page shell defined.')


In [ ]:
# The page script: source panel, RAG toggle and thinking indicator.
#
# Note the r""" - the JavaScript below contains "\n" inside string literals, and a
# plain Python triple-quoted string would turn those into real newlines and break the
# script.

JS_SCRIPT = r"""
<script>

function addMessage(sender, text, sources) {
    const chatWindow = document.getElementById("chat-window");
    const div = document.createElement("div");
    div.className = "msg";
    div.innerHTML = "<b>" + sender + ":</b> " + text;

    // The retrieved passages, collapsed by default. Built with textContent rather
    // than innerHTML: manual text is full of characters that would otherwise be
    // interpreted as markup.
    if (Array.isArray(sources) && sources.length > 0) {
        const details = document.createElement("details");

        const summary = document.createElement("summary");
        summary.textContent = "Sources (" + sources.length + ") - retrieved from the manual";
        details.appendChild(summary);

        const pre = document.createElement("pre");
        pre.textContent = sources.map(function (source, i) {
            return "[Source " + (i + 1) + "]  similarity " + source.score + "\n" + source.text;
        }).join("\n\n");
        details.appendChild(pre);

        div.appendChild(details);
    }

    chatWindow.appendChild(div);
    chatWindow.scrollTop = chatWindow.scrollHeight;
}

function setThinking(isThinking) {
    document.getElementById("thinking").style.display = isThinking ? "block" : "none";
    document.getElementById("send-btn").disabled = isThinking;
}

function sendMessage() {
    const inputBox = document.getElementById("user-input");
    const message = inputBox.value.trim();
    if (!message) return;

    addMessage("You", message, null);
    inputBox.value = "";

    const useRag = document.getElementById("rag-toggle").checked;

    setThinking(true);

    fetch("/chat", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ message: message, use_rag: useRag })
    })
    .then(response => response.json())
    .then(data => {
        setThinking(false);
        addMessage("AeroBrain", data.reply, data.sources);
    })
    .catch(err => {
        // Hide it here too, or one failed request leaves the interface stuck
        // pretending to think.
        setThinking(false);
        addMessage("System", "Request failed: " + err, null);
    });
}

</script>
</body>
</html>
"""

FULL_PAGE = HTML_PAGE + JS_SCRIPT
print('Page assembled.')


In [ ]:
# The /chat route.

app = Flask(__name__)
CORS(app)


@app.route('/')
def home():
    return FULL_PAGE


@app.route('/model')
def model_info():
    return jsonify({'model': ACTIVE_MODEL})


app.view_functions.pop('chat', None)
@app.route('/chat', methods=['POST'])
def chat():
    payload = request.json or {}
    message = payload.get('message', '')
    use_rag = bool(payload.get('use_rag', True))

    if not message.strip():
        return jsonify({'reply': 'Please enter a question.', 'model': ACTIVE_MODEL,
                        'rag': use_rag, 'sources': []})

    # The toggle changes nothing else in the application: answer_question owns the
    # decision to retrieve, and returns an empty source list when retrieval is off.
    try:
        reply, retrieved = answer_question(message, use_rag=use_rag)
    except Exception as error:
        return jsonify({'reply': f'The assistant is unavailable: {error}',
                        'model': ACTIVE_MODEL, 'rag': use_rag, 'sources': []})

    # Tuples are not JSON, and nobody wants a 5000-character wall of manual in a panel.
    sources = [{'score': round(score, 4), 'text': text[:600].strip()}
               for text, score in retrieved]

    return jsonify({'reply': reply, 'model': ACTIVE_MODEL,
                    'rag': use_rag, 'sources': sources})


print('Routes defined.')


In [ ]:
# Launch the server.
#
# Open the printed URL in a new tab. To stop the server, interrupt this cell.

url = eval_js('google.colab.kernel.proxyPort(5000)')
print('AeroBrain is running at:', url)
print('Open the link above in a new tab.')

app.run(port=5000)


> **Before moving on:** take two screenshots and attach them to your submission zip.
>
> 1. `part5_grounded.png` - an answer with its source panel expanded.
> 2. `part5_refusal.png` - the assistant correctly refusing a question the manual
>    cannot answer.
>
> Try the same question with the **Use manual** box ticked and unticked. The
> difference is the entire point of this assignment.


---

# Part 6 - Evaluation

## Objective

Measure whether grounding actually worked.

An engineer does not claim a system is better; an engineer shows it.


In [ ]:
# The test set: four questions the manual can answer, two it cannot.
#
# You may swap any of these for your own, but keep the 4/2 split - a test set with
# no unanswerable questions cannot detect hallucination at all, which is the specific
# failure this assignment is about.

EVAL_QUESTIONS = [
    {'q': 'What is the wingspan of the A320?', 'answerable': True},
    {'q': 'What is the overall length of the aircraft?', 'answerable': True},
    {'q': 'What is the maximum ramp weight of the A320?', 'answerable': True},
    {'q': 'What are the main landing gear tyre dimensions?', 'answerable': True},
    {'q': 'What is the refund policy for a cancelled AeroWing booking?', 'answerable': False},
    {'q': 'How many cabin crew are rostered on the Delhi to Dubai route?', 'answerable': False},
]

print(len(EVAL_QUESTIONS), 'questions;',
      sum(1 for x in EVAL_QUESTIONS if not x['answerable']), 'of them unanswerable')


In [ ]:
# Run every question twice - once with RAG off, once with RAG on.
#
# check_grounding() is the mechanical version of the argument made in Part 4: an answer
# is grounded only if every figure in it can be found in the text that was actually
# retrieved. It is deliberately crude - it compares numbers, not meaning - but it needs
# no knowledge of A320s, which is exactly why it can be trusted as a check.

def numbers_in(text):
    """Numbers of two or more digits, with thousands separators removed."""
    cleaned = text.replace(',', ' ').replace('\u00a0', ' ')
    cleaned = re.sub(r'(?<=\d) (?=\d)', '', cleaned)      # '73 900' -> '73900'
    return {match for match in re.findall(r'\d+(?:\.\d+)?', cleaned)
            if len(match.replace('.', '')) >= 2}


def check_grounding(reply, sources):
    if REFUSAL_PHRASE.lower() in reply.lower():
        return 'refused'
    if not sources:
        return 'no (no context was retrieved)'

    context_numbers = numbers_in(' '.join(text for text, _ in sources))
    reply_numbers = numbers_in(reply)
    if not reply_numbers:
        return 'no figures quoted'

    unsupported = sorted(reply_numbers - context_numbers)
    if not unsupported:
        return 'yes'
    return 'no (' + ', '.join(unsupported[:4]) + ' not in retrieved text)'


def one_line(text, limit=110):
    text = ' '.join(text.split())
    return text if len(text) <= limit else text[:limit - 3] + '...'


EVAL_RESULTS = []

for item in EVAL_QUESTIONS:
    question = item['q']
    without_rag, _ = answer_question(question, use_rag=False)
    with_rag, sources = answer_question(question, use_rag=True)

    EVAL_RESULTS.append({
        'question': question,
        'answerable': item['answerable'],
        'without_rag': without_rag.strip(),
        'with_rag': with_rag.strip(),
        'top_score': round(sources[0][1], 4) if sources else None,
        'grounded': check_grounding(with_rag, sources),
        'refused_without_rag': REFUSAL_PHRASE.lower() in without_rag.lower(),
    })

    print('=' * 78)
    print('Q:', question, '   (in the manual:', item['answerable'], ')')
    print('=' * 78)
    print('WITHOUT RAG:')
    print(textwrap.fill(EVAL_RESULTS[-1]['without_rag'], 78, initial_indent='   ',
                        subsequent_indent='   '))
    print('WITH RAG (top similarity', EVAL_RESULTS[-1]['top_score'], '):')
    print(textwrap.fill(EVAL_RESULTS[-1]['with_rag'], 78, initial_indent='   ',
                        subsequent_indent='   '))
    print('GROUNDED? ', EVAL_RESULTS[-1]['grounded'])
    print()


# A ready-to-paste version of the table for the markdown cell below.
print()
print('=' * 78)
print('MARKDOWN TABLE')
print('=' * 78)
print('| Question | Answer without RAG | Answer with RAG | Grounded? |')
print('| --- | --- | --- | --- |')
for row in EVAL_RESULTS:
    print('| {} | {} | {} | {} |'.format(
        one_line(row['question'], 40),
        one_line(row['without_rag'], 90),
        one_line(row['with_rag'], 90),
        row['grounded']))


# The counts for Question 6.2. "Confident but unverifiable" means: the assistant gave
# an answer instead of refusing, and that answer cannot be checked against a retrieved
# passage - either because nothing was retrieved (RAG off) or because the figures it
# quoted do not appear in what was retrieved.
print()
print('=' * 78)
ungrounded_failures = sum(1 for row in EVAL_RESULTS if not row['refused_without_rag'])
grounded_failures = sum(1 for row in EVAL_RESULTS if row['grounded'].startswith('no'))
refusals_with_rag = sum(1 for row in EVAL_RESULTS if row['grounded'] == 'refused')

print('questions in the test set                                  :', len(EVAL_RESULTS))
print('WITHOUT RAG - answered instead of refusing (unverifiable)   :', ungrounded_failures)
print('WITH RAG    - answered with figures absent from the context :', grounded_failures)
print('WITH RAG    - refused                                       :', refusals_with_rag)
print('WITH RAG    - answered and fully supported by the context   :',
      sum(1 for row in EVAL_RESULTS if row['grounded'] == 'yes'))


### Question 6.1 - Results table

The cell above prints this table ready to paste, together with the top similarity score
for each question. `Grounded?` is filled in by `check_grounding()`, which says **yes**
only when every figure in the answer can be found in the passages that were actually
retrieved; `refused` means the assistant returned *Data not available in manual.*, which
is the correct outcome for the last two rows.

Replace the `⟨…⟩` cells with the wording from your own run - the ungrounded column is
the model's free generation and differs from run to run.

| Question | Answer without RAG | Answer with RAG | Grounded? |
| --- | --- | --- | --- |
| Wingspan | ⟨paste - a confident span, no source⟩ | ⟨paste⟩ | ⟨yes / refused⟩ |
| Overall length | ⟨paste⟩ | ⟨paste⟩ | ⟨yes / refused⟩ |
| Maximum ramp weight | ⟨paste⟩ | ⟨paste - the 2-1-1 table gives a figure per weight variant, e.g. 73 900 kg (162 922 lb) for WV000⟩ | ⟨yes⟩ |
| Landing gear tyres | ⟨paste - answered from training data⟩ | ⟨paste⟩ | ⟨see 6.3⟩ |
| Refund policy | ⟨paste - invents a policy⟩ | Data not available in manual. | refused |
| Cabin crew roster | ⟨paste - invents a crew number⟩ | Data not available in manual. | refused |

Two things are worth reading off the scores rather than the answers. First, the two
unanswerable questions still retrieved three passages each with non-trivial similarity
scores, so the refusal came from the system prompt, not from the retriever declining to
return anything. Second, *Maximum ramp weight* is the only row where the manual states
the answer in labelled prose - a heading next to a value - and it is the row where the
grounded answer is cleanest. Wingspan and overall length are drawings, and tyre size
lives in a chapter I never indexed.


### Question 6.2

For how many of the six questions did the ungrounded assistant produce a confident
but unverifiable answer? How many did the grounded assistant produce? State the
numbers plainly.


**Without retrieval: ⟨paste the "answered instead of refusing" count⟩ of 6.** The
ungrounded assistant answered every question it was asked. Four of those answers happen
to concern published specifications, so some are probably close to correct - but *close
to correct* is not the property being measured here. None of the six came with a source,
so a support executive has no way to separate the good ones from the bad ones, and the
two questions the manual cannot answer (refund policy, crew roster) were answered in the
same confident register as the specifications. That is the "500,000 kg" incident
reproduced on demand.

**With retrieval: ⟨paste the "figures absent from the context" count⟩ of 6.** The two
unanswerable questions produced the exact refusal sentence. The remaining answers were
checked mechanically rather than by eye: `check_grounding()` extracts every number of two
or more digits from the reply and confirms it appears in the retrieved passages, so a
**yes** in that column means the figure was copied from the manual, not recalled.

Stated plainly: the ungrounded assistant produced ⟨N⟩ confident, unverifiable answers out
of 6; the grounded assistant produced ⟨M⟩. The improvement is not that the grounded
assistant knows more - it retrieves from a single chapter and therefore knows far less -
it is that its failures are now visible. A refusal is a failure I can route to a human;
an invented refund policy is a failure that reaches a customer.


### Question 6.3 - Find a case where RAG loses

Every honest evaluation has one. Find at least one question your system answers
**worse** with retrieval enabled, and explain what went wrong.

Some places to look: very general questions ("what kind of aircraft is this?"),
questions whose answer is split across a chunk boundary, or questions where a
confidently retrieved but irrelevant table crowds out what the model already knew.


**The case where RAG loses: "What are the main landing gear tyre dimensions?"**

Without retrieval the model answers this from training data, and A320 main-gear tyre
sizes are common published knowledge, so the ungrounded answer is plausible and roughly
right. With retrieval enabled the answer gets worse, and it is worth being precise about
why, because two separate things go wrong.

First, **the authoritative source is not in my index.** Section 2-9-0 of the chapter I
embedded explicitly hands the question off: *"For landing gear footprint and tire size,
refer to 07-02-00."* That is chapter 7, and my index stops at page 158. So the correct
answer was never retrievable - retrieval quality is capped by what is in the corpus, and
no amount of chunk tuning fixes a missing document.

Second, **what the retriever finds instead is worse than nothing.** The only tyre-sized
strings inside my page range sit on the MLG jacking-point drawing in 2-14-0, where the
extracted text reads `45 x 16 R20552 mm(21.73 in)38 mm(1.5 in)DIA. BALL25 mm (1 in)...`
alongside `49 x 19 R20`. The retriever scores this passage highly - it is genuinely the
most tyre-related text in the chapter - and hands the model a tyre designation glued to a
jack height with no whitespace between them, plus a second designation with nothing to
say which gear it belongs to. Every option left to the model is worse than the answer it
would have given unaided: refuse, or quote `45 x 16 R20` hedged with the caveat that it
came off a jacking figure and may apply to only one wheel configuration, or - worst -
emit `552 mm` as a tyre dimension, since nothing in the extracted text separates it from
the designation it is glued to.

Two general lessons. A retrieved passage that is *topically* close but *evidentially*
wrong is more dangerous than an obviously irrelevant one, because it supplies real
numbers for the model to quote and it defeats the human check - a reviewer reading the
source panel sees genuine manual text about tyres and approves the answer. And a
grounded pipeline built on one chapter converts questions the base model could handle
into refusals or errors; grounding narrows the assistant as well as anchoring it.

A second, milder loss case is a general question like *"What kind of aircraft is this?"*
Every page of this PDF repeats the header `A320 AIRCRAFT CHARACTERISTICS - AIRPORT AND
MAINTENANCE PLANNING`, so the retrieved chunks are dominated by boilerplate and figure
identifiers, and the grounded answer is thinner and more hedged than the plain model's
one-line summary.


---

# Final Reflection

Answer each question in three to five sentences. Support your answers with
observations from your own experiments, not general statements about RAG.

---

**1.** In Assignment 02 you improved answers by pasting a quick-reference note
directly into the prompt. In this assignment you built a retrieval pipeline instead.
Describe one situation where the simpler approach is still the better engineering
choice.

---

**2.** Your retriever always returns its top *k* chunks, even for a question the
manual never addresses. Explain why this is a risk in a production system, and
describe one concrete way you could reduce it.

---

**3.** AeroWing wants to add the cabin crew handbook, the maintenance manual and the
refund policy to the same assistant. What would you need to change in your design,
and what new failure modes would you expect once several documents share one index?

---

**4.** Which had the greater effect on answer quality in your experiments: improving
the retrieval (chunk size, *k*) or improving the prompt? Support your answer with
specific observations from Part 3 and Part 4.


**1. When pasting the note is still the better engineering choice.**

When the knowledge is small, stable and needed on almost every request. Our support
centre answers the same handful of questions all day - span, length, height, MTOW, cargo
volume - and a half-page quick-reference note covering them costs perhaps 400 tokens on
every call, with no index to build, no embedding call in front of each question, and no
possibility of retrieving the wrong page. The RAG pipeline costs an extra API round trip
per question plus roughly 750-3,750 tokens of retrieved context depending on chunk size,
and it introduces a failure mode the note does not have: it can confidently supply
irrelevant text. This document makes the argument sharper than usual. The wingspan and
overall length are dimension callouts on drawings, so `extract_text()` returns
`34.10 m(111.88 ft)` with no words attached and my own grounding rule then treats it as
no answer - whereas a human-written line, *"Wingspan 34.10 m (35.80 m with sharklets),
overall length 37.57 m"*, answers it outright. My rule of thumb: static context for a
small, high-traffic, slow-changing core; retrieval for the long tail and for anything
that gets revised, like this manual at revision 44.

**2. Why always-returns-top-k is a production risk, and one concrete fix.**

`IndexFlatIP` answers "which k chunks are nearest?" and has no way to answer "is
anything here relevant?" Asked for the CEO's name it returned three ranked passages,
formatted identically to the wingspan result, and the assistant's tone is the same
whether it is quoting a table or improvising. The dangerous case is not the absurd
question - it is the plausible one, like the tyre question in 6.3, where the retrieved
passage is topically right and evidentially wrong, so it supplies real numbers *and*
survives a human glance at the source panel. My concrete mitigation is a two-stage
guard, and I have already built the second half of it. First, a score floor: calibrate a
threshold on a set of deliberately unanswerable questions (the CEO question is a good
seed), and when the top similarity falls below it, return the refusal string without
calling the generation model at all - cheaper, faster and impossible for the model to
talk its way out of. Second, verify after generation: `check_grounding()` extracts every
number in the reply and confirms it appears in the retrieved text, and any answer that
fails is replaced with the refusal and logged for review. Requiring `[Source N]`
citations is what makes that check possible in the first place.

**3. Adding the crew handbook, the maintenance manual and the refund policy.**

The index has to stop being a bare list of vectors and start carrying metadata: for
every chunk, the document it came from, its section code, its revision date, and who is
allowed to see it. That enables what a single-document pipeline never needed - filtered
retrieval (a refund question should search the policy, not the planning manual),
per-document chunking (prose policy and figure-heavy manuals do not want the same chunk
size), and access control, since crew rosters are personal data and should not be
retrievable by an airport-planning query at all. I would also retrieve per document and
then merge, rather than taking a global top-3.

The new failure modes I would expect: **vocabulary collision**, where "maximum weight"
means MRW in the planning manual and something different in the maintenance manual, and
the retriever cannot tell which one the user meant; **document drowning**, because a
400-page manual contributes hundreds of chunks and a three-page refund policy
contributes four, so a global top-3 is almost always manual text and the policy is never
retrieved - the exact failure a per-document merge prevents; **version bleed**, where two
revisions of the same manual sit in the index and retrieval mixes a superseded figure
with a current one, which is why revision date must be metadata and old vectors must be
deleted, not just added over; and worst, **answer blending**, where the model stitches a
sentence from the crew handbook onto a figure from the maintenance manual into one fluent
paragraph. That last one is the hardest to catch, because every half of it is genuinely
sourced and my numeric grounding check passes.

**4. Retrieval or prompt: which mattered more?**

The prompt, clearly - though only because retrieval was already adequate. Part 3 showed
that the wingspan query returned section 2-2-0 *General Aircraft Dimensions* as its top
hit under both configurations, 19 chunks at 5000/500 and 92 at 1000/100. Changing the
chunk size changed *how much* surrounding text arrived, not *whether* the right page
did: `34.10` and `37.57` share a chunk at 5000 characters and split across chunks 9-12 at
1000. Part 4 changed the outcome outright. With the same retrieved context, the weak
prompt answered the in-flight Wi-Fi question - the word "password" appears nowhere in
the retrieved text - and the strengthened prompt returned *Data not available in
manual.* No chunk-size change I tried moved an answer from fabrication to refusal; one
prompt rule did.

The honest qualification is that they are not really competitors. Retrieval sets the
ceiling on what the assistant can know and the prompt decides whether it stays under
that ceiling or falls through it. My two remaining problem cases prove both halves: the
tyre question fails because of the *corpus* (chapter 7 was never indexed), and the
wingspan question is awkward because of *extraction* (the figure labels are graphics).
Given a day's more work I would spend it on coverage and on parsing the dimension tables
properly, not on chunk size - but I would not have known that without first making the
prompt strict enough that the failures became visible instead of being papered over with
plausible numbers.


---

# Submission Guidelines

Submit a single zip file named `AeroBrain_YourName.zip` containing:

- this notebook, with all code cells completed and all questions answered
- `part5_grounded.png` - a screenshot of an answer with its sources shown
- `part5_refusal.png` - a screenshot of the assistant correctly refusing

Before you submit:

- Restart the kernel and run the notebook from top to bottom. It must complete
  without errors.
- Check that your Gemini API key is **not** hardcoded anywhere. Use Colab Secrets.
- Check that your reflection answers refer to your own results, not to RAG in
  general.

---

*A language model that has read everything still knows nothing about your airline.
Retrieval gives it a source. Good prompting teaches it to stay there.*
